# SQLibrarian: SFT (LoRA) on Kaggle

Fine-tunes the base model (`settings.base_model`, Qwen2.5-Coder-1.5B) with LoRA on Spider + Ar-Spider
+ your own pairs, then evaluates it on the dev questions (also on a renamed, unseen schema) and exports a GGUF for CPU serving.
Cost: free (Kaggle GPU quota; about 2–3 hours for `configs/sft_run1.yaml`).

**Before you run it:**
1. On your laptop, push your own pairs once: `uv run --with datasets python -m sqlibrarian.training.format push-own`.
2. Push the branch the notebook clones (`p3-sft`).
3. Settings: Accelerator **GPU T4 x2**, Internet **on**, Persistence **Files only**.
4. Add-ons, Secrets: `HF_TOKEN` (write), `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`,
   `MLFLOW_TRACKING_PASSWORD` (DagsHub), each ticked for this notebook.
5. For the full run use **Save Version → Save & Run All (Commit)**: it keeps going after you close the tab.

| Section | What it does | Done when |
|---|---|---|
| A | Settings | it prints them |
| B | Secrets into the environment | it lists the names that are set (never the values) |
| C | Clone and install | "all imports OK" with library versions |
| D | GPU check | a Tesla T4 is listed |
| E | Train | `sft.pushed` with two Hub repos; a run in MLflow `sqlibrarian-sft` |
| F | Evaluate on the dev questions + the held-out schema | a `generate.done` line |
| G | Export GGUF | `export.pushed` |
| H | Pack predictions | `sft_predictions.zip` in Output; score it on your laptop |

## A. Settings

In [ ]:
import os

# One GPU: a single process seeing both T4s makes Trainer use nn.DataParallel,
# which TRL refuses. A 1.5B LoRA run fits on one T4. Every later step inherits this.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/OmarAmir2001/sqlibrarian.git"
BRANCH = "p3-sft"
CONFIG = "configs/sft_run1.yaml"
RUN_NAME = "sft-run1"  # must match run_name in the config
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sqlibrarian-repo"
print(BRANCH, CONFIG, RUN_NAME, sep="\n")


def run(cmd: str) -> None:
    """Run a shell command like `!cmd`, but STOP the notebook if it fails,
    so a failed step doesn't waste GPU time on the steps after it."""
    ip = get_ipython()
    ip.system(cmd)
    code = ip.user_ns.get("_exit_code", 0)
    if code != 0:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")

## B. Secrets
Read from Kaggle secrets into environment variables, where `helpers/config.py` finds them.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in [
    "HF_TOKEN",
    "MLFLOW_TRACKING_URI",
    "MLFLOW_TRACKING_USERNAME",
    "MLFLOW_TRACKING_PASSWORD",
]:
    try:
        os.environ[name] = secrets.get_secret(name)
        print("set:", name)
    except Exception:
        print("MISSING:", name)

## C. Get the code and the training libraries
`--no-deps` keeps Kaggle's GPU PyTorch. TRL and PEFT are installed on top (they may update `transformers`);
`mlflow-skinny` is the small MLflow client the trainer logs with.

In [ ]:
import subprocess
import sys

os.chdir(WORK)  # step out of an old clone, so re-running is safe
subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True
)
%cd $REPO_DIR
!git log --oneline -1

In [ ]:
!pip install -q uv
!uv pip install --system -q --no-deps -e .
!uv pip install --system -q sqlglot structlog pydantic-settings pyyaml "trl>=1.17" peft mlflow-skinny
# Kaggle ships an old torchao (0.10); new PEFT refuses to load next to it. We don't use it.
!pip uninstall -y -q torchao
# Optional, only with use_unsloth: true in the config (pins its own versions):
# !pip install -q unsloth

In [ ]:
check = (
    "import sqlibrarian.training.sft, trl, peft, transformers, torch, mlflow; "
    "print('all imports OK | trl', trl.__version__, '| peft', peft.__version__, "
    "'| transformers', transformers.__version__, '| torch', torch.__version__)"
)
subprocess.run([sys.executable, "-c", check], check=True)

## D. GPU

In [ ]:
run("nvidia-smi --query-gpu=name,memory.total --format=csv")

## E. Train
Logs a line every 10 steps and an eval loss every `eval_steps`. The best checkpoint (lowest eval loss)
is kept, merged, and pushed to two private Hub repos: `<you>/sqlibrarian-sft-run1` (merged) and
`...-lora` (adapter only).

In [ ]:
run(f"python -m sqlibrarian.training.sft --config {CONFIG}")

## F. Evaluate on the dev questions, then on the held-out (renamed) schema

In [ ]:
run(
    f"python -m sqlibrarian.evaluation.generate --model runs/{RUN_NAME}/merged --run-name {RUN_NAME}"
)
# The same questions on a renamed database the model never saw in training: does it read
# the schema, or remember names? (data/eval/schema_heldout.txt)
run(
    f"python -m sqlibrarian.evaluation.generate --model runs/{RUN_NAME}/merged"
    f" --run-name {RUN_NAME}-heldout --schema data/eval/schema_heldout.txt"
    " --questions data/eval/dev_questions_heldout.jsonl"
)

## G. Export a GGUF (Q4_K_M) for CPU serving
Builds one llama.cpp tool (a few minutes), converts, quantizes, pushes `...-gguf`.

In [ ]:
run(f"python -m sqlibrarian.training.export --model runs/{RUN_NAME}/merged --run-name {RUN_NAME}")

## H. Pack the predictions
Download `sft_predictions.zip` from Output. On your laptop (the quotes matter if the browser adds " (2)" to the name):
```bash
unzip -o "notes/sft_predictions.zip" -d /tmp/preds && cp /tmp/preds/predictions/* data/eval/predictions/
uv run --with mlflow-skinny python -m sqlibrarian.evaluation.score data/eval/predictions/<run>.jsonl --mlflow
uv run --with mlflow-skinny python -m sqlibrarian.evaluation.score --variant heldout data/eval/predictions/<run>-heldout.jsonl --mlflow
```

In [ ]:
run("cd data/eval && zip -q -r /kaggle/working/sft_predictions.zip predictions")
print("saved /kaggle/working/sft_predictions.zip")